# Machine Learning Model Comparison Lab
## Baseline → Linear Regression → Decision Tree → Random Forest → Gradient Boosting

**Problem:** Predict a continuous disease-progression measure from numeric baseline measurements.

**Dataset:** The built-in scikit-learn diabetes regression dataset. It contains 442 observations, 10 baseline features and a quantitative target.

**Learning goals**
- Understand the dataset before modelling.
- Establish a baseline that every model must beat.
- Compare different model families on exactly the same held-out test observations.
- Use cross-validation for model selection without repeatedly consulting the test set.
- Inspect residuals and feature importance instead of relying on one score.

> This is a regression teaching exercise. The dataset is not intended for clinical diagnosis or medical decision-making.


## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 2. Load and understand the data

In [ ]:
dataset = load_diabetes(as_frame=True)
X = dataset.data
y = dataset.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.describe())

In [ ]:
# Inspect data types, missing values, and summary statistics.
display(X.info())
display(X.isna().sum().to_frame("missing_values"))
display(X.describe().T)

In [ ]:
# Inspect the target distribution.
plt.figure(figsize=(8, 4))
plt.hist(y, bins=25)
plt.xlabel("Disease progression target")
plt.ylabel("Number of observations")
plt.title("Target distribution")
plt.tight_layout()
plt.show()

### Questions to answer
1. How many observations and features are available?
2. Are there missing values?
3. What is the target's range and typical value?
4. Why is this a regression problem rather than a classification problem?
5. What could go wrong if we evaluated the models on the same observations used to train them?


## 3. Create the train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

We hold the test set aside. We will use cross-validation on the training set to compare models. The test set is used once for the final comparison after the model choices are settled.

This dataset is not a time series, so a random split is reasonable for this introductory experiment. For future agricultural price forecasting, we must use chronological splitting instead.


## 4. Define evaluation metrics

We will use three regression metrics:

- **MAE:** average absolute prediction error, in target units.
- **RMSE:** square root of mean squared error; larger errors receive more weight.
- **R²:** improvement relative to predicting the mean target. It can be negative on unseen data.

No single metric tells the whole story. We will inspect all three.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

## 5. Establish a baseline

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_train_pred = baseline.predict(X_train)
baseline_test_pred = baseline.predict(X_test)

print("Baseline train:", regression_metrics(y_train, baseline_train_pred))
print("Baseline test: ", regression_metrics(y_test, baseline_test_pred))

The mean baseline ignores all input features. It gives us a minimum reference point: a useful model should do better on unseen data.

**Think about it:** Why can the baseline have a non-zero MAE even though it predicts the same value for every observation?


## 6. Linear regression

Linear regression models the target as a weighted sum of the input features:

\[
\hat y = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p
\]

It learns coefficients by minimising the sum of squared training errors. We use a pipeline so that any preprocessing is fitted within each training fold. Ordinary least squares does not require feature scaling, but the pipeline structure will be useful as we expand the experiment.


In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_train_pred = linear_model.predict(X_train)
linear_test_pred = linear_model.predict(X_test)

print("Linear train:", regression_metrics(y_train, linear_train_pred))
print("Linear test: ", regression_metrics(y_test, linear_test_pred))

In [ ]:
linear_coefficients = pd.Series(
    linear_model.named_steps["model"].coef_,
    index=X.columns
).sort_values(key=np.abs, ascending=False)

display(linear_coefficients.to_frame("coefficient"))

### Interpretation
- A positive coefficient means the model's prediction increases as that feature increases, holding other included features fixed.
- A negative coefficient means the model's prediction decreases.
- Coefficient magnitude depends on feature scale; here the model uses standardised inputs.
- These coefficients describe model associations, not causal effects.


## 7. Decision tree regression

A decision tree repeatedly splits the feature space using feature-based rules. Each leaf predicts a value based on the training observations that reach it.

A deep tree can memorise training observations. We will deliberately compare an unrestricted tree with a constrained tree to observe overfitting.


In [ ]:
deep_tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
deep_tree.fit(X_train, y_train)

shallow_tree = DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=8,
    random_state=RANDOM_STATE
)
shallow_tree.fit(X_train, y_train)

for name, model in [("Deep tree", deep_tree), ("Constrained tree", shallow_tree)]:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    print(name)
    print("  train:", regression_metrics(y_train, train_pred))
    print("  test: ", regression_metrics(y_test, test_pred))

### Investigate overfitting
Compare training and test performance. A very strong training score paired with much weaker test performance is a warning that the tree has learned details that do not generalise.


## 8. Random forest regression

A random forest trains many decision trees on bootstrapped samples and random subsets of features. Regression predictions are averaged across trees.

The averaging process often reduces the high variance of a single decision tree.


In [ ]:
forest_model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)
forest_train_pred = forest_model.predict(X_train)
forest_test_pred = forest_model.predict(X_test)

print("Random forest train:", regression_metrics(y_train, forest_train_pred))
print("Random forest test: ", regression_metrics(y_test, forest_test_pred))

## 9. Gradient boosting regression

Gradient boosting builds trees sequentially. Each new tree is fitted to improve the current ensemble, often by following the negative gradient of the chosen loss function.

The learning rate controls how much each tree contributes. The number and complexity of trees control the model's capacity.


In [ ]:
boost_model = HistGradientBoostingRegressor(
    learning_rate=0.06,
    max_iter=250,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

boost_model.fit(X_train, y_train)
boost_train_pred = boost_model.predict(X_train)
boost_test_pred = boost_model.predict(X_test)

print("Gradient boosting train:", regression_metrics(y_train, boost_train_pred))
print("Gradient boosting test: ", regression_metrics(y_test, boost_test_pred))

## 10. Compare models using cross-validation on training data

A single train/validation split can produce a noisy estimate. We now use five-fold cross-validation on the training set. Each candidate is trained and evaluated on several folds.

We compare the mean and standard deviation of MAE. Lower MAE is better. This is model selection, so the held-out test set remains untouched.


In [ ]:
candidates = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Constrained decision tree": DecisionTreeRegressor(
        max_depth=3, min_samples_leaf=8, random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=3, max_features=0.8,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient boosting": HistGradientBoostingRegressor(
        learning_rate=0.06, max_iter=250, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    )
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, estimator in candidates.items():
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1,
        return_train_score=False
    )
    cv_rows.append({
        "Model": name,
        "CV MAE mean": -scores["test_mae"].mean(),
        "CV MAE std": scores["test_mae"].std(),
        "CV RMSE mean": -scores["test_rmse"].mean(),
        "CV R2 mean": scores["test_r2"].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV MAE mean")
display(cv_results)

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(cv_results["Model"], cv_results["CV MAE mean"],
         xerr=cv_results["CV MAE std"])
plt.xlabel("Cross-validation MAE (lower is better)")
plt.title("Model comparison on training folds")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 11. Final held-out test comparison

After comparing candidates using cross-validation, we evaluate the candidates on the held-out test set. In a production workflow, we would first settle the model and hyperparameters using training/validation procedures, then evaluate the selected model on the test set once.

For this learning lab, we show the held-out scores for each predeclared candidate to make their behaviour easy to compare. Do not keep tuning based on these test scores.


In [ ]:
final_candidates = {
    "Mean baseline": baseline,
    "Linear regression": linear_model,
    "Constrained decision tree": shallow_tree,
    "Random forest": forest_model,
    "Gradient boosting": boost_model
}

test_rows = []
test_predictions = {}

for name, estimator in final_candidates.items():
    predictions = estimator.predict(X_test)
    test_predictions[name] = predictions
    test_rows.append({"Model": name, **regression_metrics(y_test, predictions)})

test_results = pd.DataFrame(test_rows).sort_values("MAE")
display(test_results)

In [ ]:
metrics_to_plot = ["MAE", "RMSE"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, metrics_to_plot):
    ordered = test_results.sort_values(metric)
    ax.barh(ordered["Model"], ordered[metric])
    ax.set_title(f"Test {metric} (lower is better)")
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 12. Error analysis

A model score tells us how much error exists; error analysis helps us understand where it comes from.

We will inspect residuals for the best cross-validation candidate. Residual = actual value − predicted value. Positive residuals mean the model underpredicted; negative residuals mean it overpredicted.


In [ ]:
best_cv_name = cv_results.iloc[0]["Model"]
best_model = candidates[best_cv_name]
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

residuals = y_test - best_pred

print("Selected using CV:", best_cv_name)
print("Test metrics:", regression_metrics(y_test, best_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(best_pred, residuals, alpha=0.7)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted value")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs predictions")

axes[1].hist(residuals, bins=20)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.show()

In [ ]:
error_table = pd.DataFrame({
    "actual": y_test,
    "predicted": best_pred,
    "residual": residuals,
    "absolute_error": np.abs(residuals)
}).sort_values("absolute_error", ascending=False)

display(error_table.head(10))

### Error-analysis questions
1. Are residuals centred around zero?
2. Does the model make larger errors for high target values?
3. Are there obvious patterns in residuals, suggesting the model misses structure?
4. Which observations have the largest absolute errors?
5. What additional data or feature engineering might help?


## 13. Permutation feature importance

Permutation importance measures how much a chosen evaluation score worsens when one feature is randomly shuffled. It is a model-agnostic diagnostic. It measures predictive reliance, not causality.

We calculate it on held-out observations for the selected model.


In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importance.plot(kind="barh")
plt.xlabel("Decrease in score when feature is shuffled")
plt.title("Permutation feature importance")
plt.tight_layout()
plt.show()

## 14. What have we learned?

Complete this comparison in your own words.

| Model | What it learns | Typical strength | Typical weakness |
|---|---|---|---|
| Mean baseline | One constant prediction | Honest reference | Ignores all features |
| Linear regression | Weighted additive relationship | Simple and interpretable | Misses complex nonlinear patterns |
| Decision tree | Hierarchical feature rules | Nonlinear and intuitive | Can be unstable and overfit |
| Random forest | Average of diverse trees | Reduces tree variance | Less transparent; can be large |
| Gradient boosting | Sequential corrections | Strong tabular performance | Needs careful tuning; can overfit |

**Do not assume a particular model must win.** The dataset, split and hyperparameters determine the result. Your job is to explain the evidence.


## 15. Reflection and submission checklist

Before considering this lab complete, write short answers to these:

1. Which model achieved the lowest cross-validation MAE?
2. Which model had the largest gap between training and test performance?
3. Did any complex model fail to beat linear regression?
4. Why is the test set not used for hyperparameter tuning?
5. What do MAE and RMSE tell us differently?
6. What patterns did you find in the residual plots?
7. Why is permutation importance not a causal explanation?
8. What would change if the observations were daily market prices over several years?

### Next experiment: agricultural market price forecasting
We will reuse the same disciplined comparison process, but change the data and validation design:
- define a specific crop, market, target statistic and forecast horizon;
- build seasonal-naive and lag-based baselines;
- create lag and rolling features without future leakage;
- compare linear, tree, random forest and boosting models;
- use walk-forward validation;
- report error by market, crop, season and forecast horizon.

The biggest conceptual change will be respecting time: a model must only learn from information that would have been available at the forecast date.
